In [5]:
import pulp

#Dataset
factories = ["F1","F2","F3","F4"]
dcs = ["DC1","DC2","DC3"]
destinations = ["D1","D2","D3","D4","D5"]


# Supply and Demand

supply = {"F1":180,"F2":140,"F3":160,"F4":120}
demand = {"D1":110,"D2":130,"D3":140,"D4":90,"D5":130}
TOTAL = sum(supply.values())  # 600

##adding the route data and constraints
#Data in the tuple (Cost, Time, CO2, Capacity)

#Factory to Distribution Centre
factory_dc = {
 ("F1","DC1"):(14,1.2,6.2,120), ("F1","DC2"):(12,1.5,7.1,120), ("F1","DC3"):(16,1.0,6.5,120),
 ("F2","DC1"):(13,1.1,6.0,120), ("F2","DC2"):(11,1.6,7.4,120), ("F2","DC3"):(15,1.2,6.6,120),
 ("F3","DC1"):(10,1.4,7.6,120), ("F3","DC2"):(9,1.7,8.2,120),  ("F3","DC3"):(12,1.3,7.4,120),
 ("F4","DC1"):(8,1.6,8.0,120),  ("F4","DC2"):(7,1.8,8.5,120),  ("F4","DC3"):(9,1.5,8.1,120),
}
#Distribution Center to Destination
dc_destination = {
 ("DC1","D1"):(9,0.9,5.8,120),  ("DC1","D2"):(11,1.0,6.1,120), ("DC1","D3"):(12,1.2,6.4,120), ("DC1","D4"):(10,1.1,6.0,120), ("DC1","D5"):(13,1.3,6.6,120),

 ("DC2","D1"):(7,0.8,6.7,120),  ("DC2","D2"):(8,0.9,6.9,120),  ("DC2","D3"):(10,1.0,7.2,120), ("DC2","D4"):(9,1.0,7.0,120),  ("DC2","D5"):(11,1.1,7.4,120),

 ("DC3","D1"):(12,0.7,6.2,120), ("DC3","D2"):(10,0.8,6.0,120), ("DC3","D3"):(9,0.9,6.3,120), ("DC3","D4"):(8,1.0,6.5,120),  ("DC3","D5"):(10,1.1,6.8,120),
}

# Factories to Destination(Direct route by train)

factory_dest = {
 ("F1","D1"):(28,2.2,4.0,300), ("F1","D2"):(30,2.4,4.2,300),
 ("F2","D1"):(27,2.3,4.1,300), ("F2","D2"):(29,2.5,4.3,300),
}


# Revised feasible caps (original caps were infeasible)

TMAX = 1367    # pallet-days (avg 2.278 days/pallet)
EMAX = 7000    # kg CO2


prob = pulp.LpProblem("NutriBox_MinCost_With_Constraints", pulp.LpMinimize)
#Ensuring that the pallets are moved as 1 and not split
# X ensures factory to destination, Y ensures DCs to destinations, Z ensures factories to destination
x = pulp.LpVariable.dicts("xFD", factory_dest.keys(), lowBound=0)     # factory->dest (rail)
y = pulp.LpVariable.dicts("yFDC", factory_dc.keys(), lowBound=0)   # factory->dc (truck)
z = pulp.LpVariable.dicts("zDCD", dc_destination.keys(), lowBound=0)   # dc->dest (truck)

# Binary for active factory->DC lanes (complexity)
u = pulp.LpVariable.dicts("uFDC", F_DC.keys(), lowBound=0, upBound=1, cat="Binary")

#Optimise the objective: Best balance (Laxed the Emission and Time constraint a bit. Time = 2.278 * 600., Emission = 7000
prob += (
    pulp.lpSum(factory_dest[a][0]  * x[a] for a in factory_dest) +
    pulp.lpSum(factory_dc[a][0] * y[a] for a in factory_dc) +
    pulp.lpSum(dc_destination[a][0] * z[a] for a in dc_destination)
)

# Capacity constraints
for a in factory_dest:  prob += x[a] <= factory_dest[a][3]
for a in factory_dc: prob += y[a] <= factory_dc[a][3]
for a in dc_destination: prob += z[a] <= dc_destination[a][3]

# Supply constraints 
for f in factories:
    prob += (
        pulp.lpSum(y[(f,k)] for k in dcs) +
        pulp.lpSum(x[(f,d)] for d in destinations if (f,d) in factory_dest)
        == supply[f]
    )

# Demand constraints 
for d in dests:
    prob += (
        pulp.lpSum(z[(k,d)] for k in dcs) +
        pulp.lpSum(x[(f,d)] for f in factories if (f,d) in factory_dest)
        == demand[d]
    )

# DC flow conservation
for k in dcs:
    prob += pulp.lpSum(y[(f,k)] for f in factories) == pulp.lpSum(z[(k,d)] for d in destinations)

# DC2 inbound limit
prob += pulp.lpSum(y[(f,"DC2")] for f in factories) <= 150

# Risk diversification: each DC outbound <= 270
for k in dcs:
    prob += pulp.lpSum(z[(k,d)] for d in destinations) <= 270

# Minimum rail usage
prob += pulp.lpSum(x[a] for a in factory_dest) >= 120

# Time cap
prob += (
    pulp.lpSum(F_D[a][1]  * x[a] for a in factory_dest) +
    pulp.lpSum(F_DC[a][1] * y[a] for a in factory_dc) +
    pulp.lpSum(DC_D[a][1] * z[a] for a in dc_destination)
    <= TMAX
)

# Emissions cap
prob += (
    pulp.lpSum(F_D[a][2]  * x[a] for a in F_D) +
    pulp.lpSum(F_DC[a][2] * y[a] for a in F_DC) +
    pulp.lpSum(DC_D[a][2] * z[a] for a in DC_D)
    <= EMAX
)

# Complexity: max 8 active factory->DC lanes
M = 120
for a in F_DC:
    prob += y[a] <= M * u[a]
prob += pulp.lpSum(u[a] for a in F_DC) <= 8

# Solve with CBC
prob.solve(pulp.PULP_CBC_CMD(msg=True))



1

In [10]:
if pulp.LpStatus[prob.status] == "Optimal":
    print("-" * 75)
    print(f"{"ROUTE TYPE":<20} | {"PATH":<12} | {"PALLETS":<8} | {"COST":<8} | {"TIME":<6} | {"CO2":<6}")
    print("-" * 75)
    
    # Track metrics for validation
    total_pallets = 0
    total_time = 0
    total_co2 = 0
    total_financial_cost = 0

    # Factory to DC
    for r in factory_dc:
        val = pulp.value(y[r])
        if val > 0:
            c, t, e, _ = factory_dc[r]
            print(f"{"Factory -> DC":<20} | {str(r):<12} | {int(val):<8} | ${c*val:<7} | {t:<6} | {e:<6}")
            total_pallets += val
            total_time += val * t
            total_co2 += val * e
            total_financial_cost += c * val

    # DC to Destination
    for r in dc_destination:
        val = pulp.value(z[r])
        if val > 0:
            c, t, e, _ = dc_destination[r]
            print(f"{"DC -> Destination":<20} | {str(r):<12} | {int(val):<8} | ${c*val:<7} | {t:<6} | {e:<6}")
            # Note: We don't add to total_pallets here to avoid double counting (it's the same 600 pallets)
            total_time += val * t
            total_co2 += val * e
            total_financial_cost += c * val

    # Direct destination route
    for r in factory_dest:
        val = pulp.value(x[r])
        if val > 0:
            c, t, e, _ = factory_dest[r]
            print(f"{"Direct destination(rail)":<20} | {str(r):<12} | {int(val):<8} | ${c*val:<7} | {t:<6} | {e:<6}")
            total_pallets += val
            total_time += val * t
            total_co2 += val * e
            total_financial_cost += c * val

    print("-" * 75)
    print(f"FINAL RESULTS (Optimised to achive the best of both:")
    print(f"Total Time Optimised : {pulp.value(prob.objective):,.2f} Days")
    print(f"Total Financial Cost: ${total_financial_cost:,.2f}")
    print(f"Average Lead Time:    {total_time/600:.2f} Days (Limit: 2.1)")
    print(f"Total Emissions:      {total_co2:.1f} kg CO2 (Limit: 3900)")
    print(f"Active F->DC Lanes:   {sum(pulp.value(u[r]) for r in u):.0f} (Limit: 8)")
else:
    print("No optimal solution found. Check constraints.")

    ### Adjust this to show you the infeasibility.

---------------------------------------------------------------------------
ROUTE TYPE           | PATH         | PALLETS  | COST     | TIME   | CO2   
---------------------------------------------------------------------------
Factory -> DC        | ('F1', 'DC3') | 80       | $1280.0  | 1.0    | 6.5   
Factory -> DC        | ('F2', 'DC1') | 120      | $1560.0  | 1.1    | 6.0   
Factory -> DC        | ('F3', 'DC1') | 60       | $600.0   | 1.4    | 7.6   
Factory -> DC        | ('F3', 'DC3') | 100      | $1200.0  | 1.3    | 7.4   
Factory -> DC        | ('F4', 'DC2') | 30       | $210.0   | 1.8    | 8.5   
Factory -> DC        | ('F4', 'DC3') | 90       | $810.0   | 1.5    | 8.1   
DC -> Destination    | ('DC1', 'D2') | 90       | $990.0   | 1.0    | 6.1   
DC -> Destination    | ('DC1', 'D4') | 90       | $900.0   | 1.1    | 6.0   
DC -> Destination    | ('DC2', 'D2') | 0        | $5.39075536e-12 | 0.9    | 6.9   
DC -> Destination    | ('DC2', 'D3') | 20       | $200.0   | 1.0    | 7.